# Example: ClothCo
ClothCo is capable of manufacturing three types of clothing: shirts, shorts, and pants. Each type of clothing requires that ClothCo have the appropriate type of machine available. The machines can be rented at a fixed weekly cost. The manufacture of each type of clothing also requires some amount of cloth and labor, and returns some profit, indicated below. Each week, 150 hours of labor and 160 sq yd of cloth are available. How should ClothCo tune its production to maximize profits? Note: If we don't produce a particular item, we can save on the rental cost!

| Clothing item | Labor per item | Cloth per item (sq yd) | Profit per item | Machine rental |
|:------:|:-------:|:-:|:---:|:--------:|
| Shirt  | 3 hours | 4 | \$6 | \$200/wk |
| Shorts | 2 hours | 3 | \$4 | \$150/wk |
| Pants  | 6 hours | 4 | \$7 | \$100/wk |

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. All required packages are included; no external data files are needed.

Let $x_1,x_2,x_3\ge0$ be weekly production of shirts, shorts, and pants. Production quantities are continuous in this example. Let $z_i\in\{0,1\}$ indicate whether machine $i$ is rented. Net profit subtracts the weekly machine rentals from production profit.

The linking constraints $x_i\le54z_i$ force production to zero when a machine is not rented. The bound 54 is valid for every product: even shorts, which use the least cloth per item, cannot exceed $160/3<54$ items. Renting a machine permits production but does not require it.

In [ ]:
using JuMP, HiGHS

m = Model(HiGHS.Optimizer)
set_silent(m)

@variable(m, x[1:3] >= 0)
@variable(m, z[1:3], Bin)

@constraint(m, 3x[1] + 2x[2] + 6x[3] <= 150)  # labor budget
@constraint(m, 4x[1] + 3x[2] + 4x[3] <= 160)  # cloth budget
@constraint(m, x .<= 54*z)                   # if x>0 then z=1

@objective(m, Max, 6x[1] + 4x[2] + 7x[3] - 200z[1] - 150z[2] - 100z[3])

optimize!(m)
if !is_solved_and_feasible(m; allow_local = false)
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end

xopt = value.(x)
println(xopt[1], " shirts")
println(xopt[2], " shorts")
println(xopt[3], " pants" )
println()
println("\$", objective_value(m), " of net profit")